# Süt Verimi Tahmini — Yem Verisiyle (Rwanda Veri Seti)

**Amaç:** 03'te "verim tahmini için yem/sağlık verisi lazım" demiştik. Bu veri
setinde (Rwanda, 2020-2021, 106 inek) yem miktarı, vücut ağırlığı gibi ADİL
özellikler var. Bu sefer bunlarla süt verimini tahmin etmeyi deniyoruz.

**Dikkat:** Veride "gapmilk", "potentialmilk" gibi doğrudan süt veriminden
hesaplanmış sütunlar var — bunlar data leakage riski taşır (04'te öğrendik),
o yüzden atılacak.

In [4]:
from google.colab import files
yuklenen = files.upload()

Saving Specific data recorded on individual cows under lactation in Rwanda 2020-2021.xlsx to Specific data recorded on individual cows under lactation in Rwanda 2020-2021 (1).xlsx


In [8]:
import pandas as pd

# Gerçek başlık 10. satırda (üstünde çalışma başlığı, yazarlar var)
df = pd.read_excel(
    list(yuklenen.keys())[0],
    sheet_name="Raw data",
    header=10
)

print("Boyut:", df.shape)
print("Sütunlar:", df.columns.tolist())
df.head()

Boyut: (96, 43)
Sütunlar: ['sites', 'LabN°', 'cowbreed', 'cowageinyears', 'parity', 'Bodyweight', 'MW', 'DMIR kg', 'DM served', 'leftover', 'daysinmilk', 'lactationperiod', 'Ass.calfmilk', 'hand-milked yield', 'Total milk performance', 'gapmilk', 'potentialmilk', '%gapmilk', 'waterday', 'waterrequi.', 'gapwater', '%watergap', 'DMfeeds', 'MEfeeds', 'NDF feeds', 'DMIindex', 'DMIcapacity (kgDM)', 'DMI gap', '%gapDMI', 'MEIntake', 'MW*0.589=Energyformaintenance', '5.023*peakMilk', 'MEmaint+peakmilk', 'gapME', '%MEgap', '%Protein', 'Protein/content/gr/kg', 'Cpintakeingr', 'CPmaint=6.27*MW', 'CPmilk', 'TotalreqCP', 'gapCP', '%CP gap']


,sites,LabN°,cowbreed,cowageinyears,parity,Bodyweight,MW,DMIR kg,DM served,leftover,...,gapME,%MEgap,%Protein,Protein/content/gr/kg,Cpintakeingr,CPmaint=6.27*MW,CPmilk,TotalreqCP,gapCP,%CP gap
0,1,224,Cross,8,4.0,432,94.757329,15.12,16.15724,8.351187,...,84.958492,64.776146,10.02,100.2,782.166541,594.128452,1230,1824.128452,1041.961911,57.121082
1,1,225,Cross,NaN,3.0,520,108.893638,18.20,13.79940,0.973090,...,91.097029,65.310323,10.28,102.8,1318.544707,682.763111,1230,1912.763111,594.218404,31.065969
2,1,227,Cross,6,2.0,490,104.147071,17.15,11.27400,0.938264,...,72.415241,52.978637,9.06,90.6,936.417648,653.002135,1230,1883.002134,946.584486,50.269964
3,1,228,exotic,6,3.0,496,105.102067,17.36,11.73000,2.249274,...,108.777363,79.254842,7.89,78.9,748.029309,658.989960,1230,1888.989960,1140.960652,60.400567
4,1,230,exotic,7,4.0,490,104.147071,17.15,14.81620,5.442694,...,83.040944,60.752350,10.96,109.6,1027.336203,653.002135,1230,1883.002134,855.665932,45.441581


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 43 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   sites                          96 non-null     int64  
 1   LabN°                          96 non-null     int64  
 2   cowbreed                       96 non-null     object 
 3   cowageinyears                  94 non-null     object 
 4   parity                         93 non-null     float64
 5   Bodyweight                     96 non-null     int64  
 6   MW                             96 non-null     float64
 7   DMIR kg                        96 non-null     float64
 8   DM served                      96 non-null     float64
 9   leftover                       96 non-null     float64
 10  daysinmilk                     96 non-null     int64  
 11  lactationperiod                96 non-null     object 
 12  Ass.calfmilk                   96 non-null     int64

In [10]:
# Sadece adil özellikleri + hedefi seç (leakage ve kimlik sütunlarını dışarıda bırak)
adil_sutunlar = ["cowbreed", "cowageinyears", "parity", "Bodyweight",
                 "DMIR kg", "DM served", "leftover", "daysinmilk",
                 "lactationperiod", "DMfeeds", "NDF feeds", "waterday",
                 "Total milk performance"]

df_temiz = df[adil_sutunlar].copy()

# Eksik satırları at (az sayıda)
df_temiz = df_temiz.dropna()

print("Temiz boyut:", df_temiz.shape)
print("Sütunlar:", df_temiz.columns.tolist())
df_temiz.head()

Temiz boyut: (91, 13)
Sütunlar: ['cowbreed', 'cowageinyears', 'parity', 'Bodyweight', 'DMIR kg', 'DM served', 'leftover', 'daysinmilk', 'lactationperiod', 'DMfeeds', 'NDF feeds', 'waterday', 'Total milk performance']


,cowbreed,cowageinyears,parity,Bodyweight,DMIR kg,DM served,leftover,daysinmilk,lactationperiod,DMfeeds,NDF feeds,waterday,Total milk performance
0,Cross,8,4.0,432,15.120,16.15724,8.351187,270,Late,28.53,66.41,20,3.0
2,Cross,6,2.0,490,17.150,11.27400,0.938264,240,Late,18.70,56.89,20,2.0
3,exotic,6,3.0,496,17.360,11.73000,2.249274,30,Peak,17.88,62.78,20,15.0
4,exotic,7,4.0,490,17.150,14.81620,5.442694,30,Peak,19.68,62.73,40,17.5
5,Cross,7.5,3.0,447,15.645,14.12625,6.713315,90,Peak,20.05,72.36,20,10.6


In [13]:
# cowageinyears sütununu sayıya çevir, hatalı metinleri NaN yap ve temizle
df_temiz['cowageinyears'] = pd.to_numeric(df_temiz['cowageinyears'], errors='coerce')
df_temiz = df_temiz.dropna(subset=['cowageinyears'])

# Metin sütunlarını (ırk, laktasyon dönemi) sayıya çevir
df_hazir = pd.get_dummies(df_temiz, columns=["cowbreed", "lactationperiod"], drop_first=True)

# Özellik (X) ve hedef (y)
X = df_hazir.drop("Total milk performance", axis=1)
y = df_hazir["Total milk performance"]

print("X boyutu:", X.shape)
print("Sütunlar:", X.columns.tolist())

X boyutu: (64, 13)
Sütunlar: ['cowageinyears', 'parity', 'Bodyweight', 'DMIR kg', 'DM served', 'leftover', 'daysinmilk', 'DMfeeds', 'NDF feeds', 'waterday', 'cowbreed_exotic', 'lactationperiod_Mid', 'lactationperiod_Peak']


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_r2 = r2_score(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)

print("--- YEM VERİSİYLE (adil özellikler) ---")
print("Linear Regression → MAE:", round(lr_mae, 2), "| R²:", round(lr_r2, 3))
print("Random Forest      → MAE:", round(rf_mae, 2), "| R²:", round(rf_r2, 3))
print()
print("Hatırlatma → 03 (Mendeley, yemsiz): R² 0.097")

--- YEM VERİSİYLE (adil özellikler) ---
Linear Regression → MAE: 1.49 | R²: 0.802
Random Forest      → MAE: 2.09 | R²: 0.619

Hatırlatma → 03 (Mendeley, yemsiz): R² 0.097


In [15]:
import joblib

# En iyi modeli (Linear Regression) kaydet
joblib.dump(lr, "sut_verimi_modeli.pkl")
print("Model kaydedildi: sut_verimi_modeli.pkl")

# Tek bir inek üzerinde test
ornek = X_test.iloc[[0]]
tahmin = lr.predict(ornek)
print("\nGerçek verim:", round(y_test.iloc[0], 2))
print("Model tahmini:", round(tahmin[0], 2))

Model kaydedildi: sut_verimi_modeli.pkl

Gerçek verim: 11.0
Model tahmini: 13.37


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor

# Ölçekleme
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Birkaç modeli birden dene
modeller = {
    "Linear (ölçekli)": LinearRegression(),
    "Ridge (ölçekli)": Ridge(alpha=1.0),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

print("--- İYİLEŞTİRME DENEMELERİ ---")
for isim, m in modeller.items():
    m.fit(X_train_s, y_train)
    pred = m.predict(X_test_s)
    print(f"{isim:20} → MAE: {mean_absolute_error(y_test, pred):.2f} | R²: {r2_score(y_test, pred):.3f}")

print()
print("Önceki en iyi → Linear Regression: R² 0.802")

--- İYİLEŞTİRME DENEMELERİ ---
Linear (ölçekli)     → MAE: 1.49 | R²: 0.802
Ridge (ölçekli)      → MAE: 1.59 | R²: 0.788
GradientBoosting     → MAE: 2.14 | R²: 0.622

Önceki en iyi → Linear Regression: R² 0.802


##  Kapanış — Yem Verisiyle Gerçek Verim Tahmini

Bu notebook'ta, Rwanda süt ineği veri setiyle (91 kayıt), yem miktarı ve hayvan
özelliklerine dayanarak süt verimini tahmin eden **gerçek ve kullanılabilir** bir
model kurduk.

**İzlenen adımlar**
1. Veriyi doğru başlık satırıyla okuma (`header=10`)
2. Kimlik ve data leakage riski taşıyan (gapmilk, potentialmilk vb.) sütunları eleme
3. Sadece adil özellikleri seçme (yem, ağırlık, ırk, laktasyon dönemi)
4. Metin sütunlarını One-Hot Encoding ile sayıya çevirme
5. İki model eğitip karşılaştırma, ardından iyileştirme denemeleri

**Sonuçlar**
| Model | MAE | R² |
|---|---|---|
| Linear Regression | 1.49 | 0.802 |
| Random Forest | 2.09 | 0.619 |
| GradientBoosting (ölçekli) | 2.14 | 0.622 |

**Üç notebook'un birlikte anlattığı hikâye**
- **03:** Yemsiz veriyle (ırk, parity) → R² 0.10 → özellikler yetersizdi.
- **04:** Süt bileşenleri eklendi → R² 0.99 → ama bu data leakage, sahte başarı.
- **05:** Doğru veri (yem, ağırlık) bulundu → R² 0.80 → **gerçek ve adil tahmin.**

**Temel Bulgular**
1. Bu sefer yüksek R², bir kopya değil gerçek bir başarıdır — çünkü tüm özellikler
   (yem miktarı, vücut ağırlığı, ırk, laktasyon dönemi) süt sağılmadan ÖNCE bilinen,
   hedefe sızmayan adil değişkenlerdir. Bu, 03'teki "daha iyi veri gerekiyor"
   bulgusunu doğruladı.
2. İyileştirme denemeleri (ölçekleme, Ridge, GradientBoosting) R²'yi 0.80'in üstüne
   çıkaramadı. Bu, sınırın modelde değil **veri miktarında** (91 satır) olduğunu
   gösterdi. Daha yüksek doğruluk için daha fazla veri gerekir.
3. Basit Linear Regression, karmaşık modelleri (GradientBoosting) geçti — küçük ve
   büyük ölçüde doğrusal veride "en karmaşık model her zaman en iyisi değildir"
   ilkesi bir kez daha doğrulandı.

**Sonuç:** Eğitilen model `sut_verimi_modeli.pkl` olarak kaydedildi ve projenin
backend'ine entegre edilmeye hazır — bu, projenin ilk çalışan gerçek ML modelidir.